# Data Consolidation
This notebook reads multiple CSV files from the `data/raw/` directory and consolidates them into 3 main tables for easier analysis:
1. **dim_customers_accounts**: Combined information about customers and their bank accounts.
2. **fact_transactions**: Transaction history with detailed types and branch information.
3. **fact_loans**: Loan data with status descriptions.

In [11]:
import pandas as pd
import os

# Set the path to the raw data
raw_data_path = '../data/raw/'
processed_data_path = '../data/processed/'

# Load all CSV files
account_statuses = pd.read_csv(os.path.join(raw_data_path, 'account_statuses.csv'))
account_types = pd.read_csv(os.path.join(raw_data_path, 'account_types.csv'))
accounts = pd.read_csv(os.path.join(raw_data_path, 'accounts.csv'))
addresses = pd.read_csv(os.path.join(raw_data_path, 'addresses.csv'))
branches = pd.read_csv(os.path.join(raw_data_path, 'branches.csv'))
customer_types = pd.read_csv(os.path.join(raw_data_path, 'customer_types.csv'))
customers = pd.read_csv(os.path.join(raw_data_path, 'customers.csv'))
loan_statuses = pd.read_csv(os.path.join(raw_data_path, 'loan_statuses.csv'))
loans = pd.read_csv(os.path.join(raw_data_path, 'loans.csv'))
transaction_types = pd.read_csv(os.path.join(raw_data_path, 'transaction_types.csv'))
transactions = pd.read_csv(os.path.join(raw_data_path, 'transactions.csv'))

## 1. Create dim_customers_accounts
We join customers with their addresses, types, and then link them to their accounts.

In [12]:
# Join customers with types and addresses
customers_full = customers.merge(customer_types, on='CustomerTypeID', how='left', suffixes=('', '_Type'))
customers_full = customers_full.merge(addresses, on='AddressID', how='left')

# Join accounts with types and statuses
accounts_full = accounts.merge(account_types, on='AccountTypeID', how='left', suffixes=('', '_Type'))
accounts_full = accounts_full.merge(account_statuses, on='AccountStatusID', how='left', suffixes=('', '_Status'))

# Consolidate Customer and Account information
dim_customers_accounts = accounts_full.merge(customers_full, on='CustomerID', how='left', suffixes=('_Account', '_Customer'))

print(f"dim_customers_accounts shape: {dim_customers_accounts.shape}")
dim_customers_accounts.head()

dim_customers_accounts shape: (1758, 17)


,AccountID,CustomerID,AccountTypeID,AccountStatusID,Balance,OpeningDate,TypeName_Account,StatusName,FirstName,LastName,DateOfBirth,AddressID,CustomerTypeID,TypeName_Customer,Street,City,Country
0,200094,10123,3,1,48348.54,2018-06-12 00:00:00.000000,Payroll,Active,Cecille,Spencer,1993-12-05 00:00:00.000000,499,2,Small Business,Girard,Menasha,United States
1,201108,10077,3,1,35001.41,2019-10-30 00:00:00.000000,Payroll,Active,Dario,Campbell,1964-04-16 00:00:00.000000,12,2,Small Business,Judson,Bellwood,United States
2,201453,10321,3,2,57081.03,2020-05-24 00:00:00.000000,Payroll,Inactive,Mack,Goodwin,1989-09-18 00:00:00.000000,246,3,Large Enterprise,Michigan,Edina,United States
3,200581,10871,5,1,63164.33,2021-01-27 00:00:00.000000,Youth,Active,Tiffiny,Vinson,1964-05-09 00:00:00.000000,362,2,Small Business,Starview,Manhattan,United States
4,200003,10765,1,1,58739.64,2018-09-12 00:00:00.000000,Checking,Active,Junior,Witt,1992-07-09 00:00:00.000000,870,1,Individual,Murray,Marshalltown,United States


## 2. Create fact_transactions
We join transactions with their types and branch information.

In [13]:
# Join branches with their addresses
branches_full = branches.merge(addresses, on='AddressID', how='left', suffixes=('', '_Branch'))

# Join transactions with types and branches
fact_transactions = transactions.merge(transaction_types, on='TransactionTypeID', how='left', suffixes=('', '_Type'))
fact_transactions = fact_transactions.merge(branches_full, on='BranchID', how='left', suffixes=('', '_Branch'))

print(f"fact_transactions shape: {fact_transactions.shape}")
fact_transactions.head()

fact_transactions shape: (50995, 14)


,TransactionID,AccountOriginID,AccountDestinationID,TransactionTypeID,Amount,TransactionDate,BranchID,Description,TypeName,BranchName,AddressID,Street,City,Country
0,3022681,201164,200868,2,855.17,2023-04-20 02:00:00.000000,41,Transaction 22681,Withdrawal,Branch 41,173,Houston,Lake Oswego,United States
1,3037846,200138,201402,2,806.20,2021-08-10 15:00:00.000000,43,Transaction 37846,Withdrawal,Branch 43,742,Fort Mason,Louisville,United States
2,3045293,201002,201180,1,1229.44,2020-08-16 03:00:00.000000,5,Transaction 45293,Deposit,Branch 5,796,Getz,Steubenville,United States
3,3017397,201066,201144,4,4441.60,2021-10-10 06:00:00.000000,14,Transaction 17397,Payment,Branch 14,491,Forest Knolls,Kinston,United States
4,3016750,200289,201413,3,2526.20,2022-07-28 00:00:00.000000,37,Transaction 16750,Transfer,Branch 37,265,Thornburg,University Place,United States


## 3. Create fact_loans
We join loans with their status names.

In [14]:
# Join loans with statuses
fact_loans = loans.merge(loan_statuses, on='LoanStatusID', how='left', suffixes=('', '_Status'))

print(f"fact_loans shape: {fact_loans.shape}")
fact_loans.head()

fact_loans shape: (333, 8)


,LoanID,AccountID,LoanStatusID,PrincipalAmount,InterestRate,StartDate,EstimatedEndDate,StatusName
0,400230,200876,1,76958.56,0.0547,2022-11-20 00:00:00.000000,2026-08-06 00:00:00.000000,Active
1,400307,200789,1,29013.67,0.0321,2022-02-22 00:00:00.000000,2025-12-08 00:00:00.000000,Active
2,400233,201275,1,48596.76,0.1017,2021-11-21 00:00:00.000000,2023-07-30 00:00:00.000000,Active
3,400100,200070,1,9191.43,0.0999,2021-08-14 00:00:00.000000,2023-09-18 00:00:00.000000,Active
4,400141,200808,1,76322.83,0.0906,2021-06-04 00:00:00.000000,2024-10-23 00:00:00.000000,Active


## Save the consolidated tables

In [15]:
# Create processed directory if it doesn't exist
if not os.path.exists(processed_data_path):
    os.makedirs(processed_data_path)

dim_customers_accounts.to_csv(os.path.join(processed_data_path, 'dim_customers_accounts.csv'), index=False)
fact_transactions.to_csv(os.path.join(processed_data_path, 'fact_transactions.csv'), index=False)
fact_loans.to_csv(os.path.join(processed_data_path, 'fact_loans.csv'), index=False)

print("Consolidated tables saved to data/processed/")

Consolidated tables saved to data/processed/
